# README

- **Author**: 楊喬茵（B12303061；AI 協助資料蒐集與整理）
- **Created At**: 2026-11-04
- **Last Modified At**: 2026-11-04

---

## What does this file do?

- 補查臺北市、高雄市直轄市議員候選人（199 人）在 G1 年齡、G3 學歷、G5 首次參選任何民選公職仍有缺口者：79 人、208 格。觀察單位是「候選人 × 缺口欄位」。G2 兩市無缺值；G6 不在本輪範圍。
- **所有新聞與網頁都讀完全文才判斷**（依 `FINDING-RULES.md`「網路資料與新聞須讀完全文」）。關鍵字只用來找文章。
- 判斷規則沿用 [縣市首長補查筆記本](mayor_gaps_20261104.ipynb) 第 4 節的 G5 判定標準（楊喬茵 2026-11-04 修訂），另加本輪規則（第 4 節）。每格的 `decision_basis` 註明依據，所有補上的值 `check_method` 皆為「手動查核」。
- 中選會歷屆候選人檔的同名紀錄，只有在身分有獨立佐證時才採用（同選區、與人物頁出生日相符、或新聞明載曾任該職）；同名但另一縣市或年齡不符者視為他人。
- 本筆記本**不覆寫** `candidates_g1_g6_current.csv`。

---

## What does this file take?

- **Source Data Sets** (one entry per actual file；SHA-256 均為換行統一為 LF 後的值):
  1. `data/processed/candidates_g1_g6_current.csv`
     - Input key: `g1_g6_current`
     - Description: G1–G6 主表，1,583 人；鍵為 `candidate_key`。
     - Documentation: `data/processed/g1_g6_20261103_documentation.ipynb`
     - SHA-256: `d8695b306bd7eadbf0de5c8bb2f368c717d23d4ef7ae55cf3cf4b8fa1afbb59c`
  2. `data/raw/council_tk_gaps_20261104/manifest.json` 及其列出的成功抓取檔
     - Input key: `council_captures`
     - Description: 以 `data/code/collect_council_tk_20261104.py` 下載的 political.tw 人物頁、維基百科、自由時報站內搜尋與文章、中央社、聯合報、新頭殼等；失敗（如 404）者保留錯誤紀錄，不當作來源。網址清單見 `data/code/council_tk_sources_20261104.json`，人員對照見 `data/code/council_tk_people_20261104.json`。
     - Documentation: 本筆記本與 manifest
     - SHA-256: `2010208bcd78ce0f84c01ad91b5bf308c4b0c61d997ef2a92ff5e9de9cfeea74`
  3. `data/raw/cec_all_offices_manifest.json` 及其列出的中選會 1994–2024 歷屆候選人檔
     - Input key: `cec_all_offices`
     - Documentation: `data/raw/sources_2026_documentation.ipynb`
     - SHA-256: `9cc85f7f913f5de717ba92ec7698c73cedb29df506fba59a69933d1ac3517231`
  4. `data/raw/cross_office_round8_20261019/2022_village_elbase.csv`
     - Input key: `cec_2022_geo_base`
     - Description: 中選會 2022 行政區代碼表，用於把歷屆紀錄的代碼轉成縣市、區、里名。
     - Documentation: `data/raw/cross_office_round8_20261019_documentation.ipynb`
     - SHA-256: `4d0afbae084ccd77785ebe9a878eadb7e97e828555926d92866dddbfb4b14e79`

---

## What does this file output?

- `data/processed/council_tk_g1_g5_gap_review_20261104.csv`
  - Output key: `gap_review`
  - Description: 208 列，每列一位候選人的一個缺口欄位，含補上值、判斷類別、`decision_basis`、`check_method`、原文摘錄、來源網址與原始檔、中選會紀錄位置、說明。
  - Role: preparation（補查紀錄；未併入主表）
  - Documentation: 本筆記本
- `data/processed/council_tk_cec_name_scan_20261104.csv`
  - Output key: `cec_name_scan`
  - Description: 79 人在中選會歷屆檔的同名命中（含原住民族名部分比對），附行政區名稱與採用與否。
  - Role: preparation（G1／G3／G5 證據）
  - Documentation: 本筆記本
- `data/processed/council_tk_manual_check_links_20261104.csv`
  - Output key: `manual_check_links`
  - Description: 仍未知或身分無法確認的缺口，附疑慮與連結。
  - Role: worklist（手動查核用）
  - Documentation: 本筆記本


In [1]:
from pathlib import Path
import hashlib, json, csv, re, unicodedata
import pandas as pd
from bs4 import BeautifulSoup


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()
# SHA-256 after CRLF->LF: identical to the bytes Git stores.
sha = lambda rel: hashlib.sha256((PROJECT_ROOT / rel).read_bytes().replace(b"\r\n", b"\n")).hexdigest()
norm = lambda s: re.sub(r"\s+", "", unicodedata.normalize("NFKC", s))
INPUTS = {
    "g1_g6_current": "data/processed/candidates_g1_g6_current.csv",
    "council_captures": "data/raw/council_tk_gaps_20261104/manifest.json",
    "cec_all_offices": "data/raw/cec_all_offices_manifest.json",
    "cec_2022_geo_base": "data/raw/cross_office_round8_20261019/2022_village_elbase.csv",
}
EXPECTED_INPUT_SHA256 = {
    "g1_g6_current": "d8695b306bd7eadbf0de5c8bb2f368c717d23d4ef7ae55cf3cf4b8fa1afbb59c",
    "council_captures": "2010208bcd78ce0f84c01ad91b5bf308c4b0c61d997ef2a92ff5e9de9cfeea74",
    "cec_all_offices": "9cc85f7f913f5de717ba92ec7698c73cedb29df506fba59a69933d1ac3517231",
    "cec_2022_geo_base": "4d0afbae084ccd77785ebe9a878eadb7e97e828555926d92866dddbfb4b14e79",
}
OUTPUTS = {
    "gap_review": "data/processed/council_tk_g1_g5_gap_review_20261104.csv",
    "cec_name_scan": "data/processed/council_tk_cec_name_scan_20261104.csv",
    "manual_check_links": "data/processed/council_tk_manual_check_links_20261104.csv",
}
for key, rel in INPUTS.items():
    assert sha(rel) == EXPECTED_INPUT_SHA256[key], f"Input changed: {rel}"
captures = json.loads((PROJECT_ROOT / INPUTS["council_captures"]).read_text(encoding="utf-8"))
ok = [c for c in captures if c["status"] == "ok"]
for c in ok:
    assert sha(c["file"]) == c["sha256_lf"], c["file"]
CAPTURE = {c["label"]: c for c in ok}
cec_manifest = json.loads((PROJECT_ROOT / INPUTS["cec_all_offices"]).read_text(encoding="utf-8"))
for m in cec_manifest["members"]:
    raw = (PROJECT_ROOT / m["path"]).read_bytes()
    assert m["sha256"] in {hashlib.sha256(raw).hexdigest(), sha(m["path"])}, m["path"]
people = json.loads((PROJECT_ROOT / "data/code/council_tk_people_20261104.json").read_text(encoding="utf-8"))
print(len(ok), "captures verified;", len(captures) - len(ok), "failed fetches kept as errors only;",
      len(cec_manifest["members"]), "CEC entries verified;", len(people), "people")


370 captures verified; 66 failed fetches kept as errors only; 146 CEC entries verified; 79 people


## 1. 缺口

In [2]:
main = pd.read_csv(PROJECT_ROOT / INPUTS["g1_g6_current"], dtype=str, keep_default_na=False)
FIELDS = {"G1": "G1_age_2026_minus_birth_year", "G3": "G3_education_level", "G5": "G5_first_time_any_elected_office"}
council = main[main.county_city.isin(["臺北市", "高雄市"]) & main.office_type.eq("直轄市議員")]
assert len(council) == 199 and (council.G2_sex != "").all()
gaps = pd.DataFrame([(r.candidate_key, r.candidate_name, r.county_city, r.electoral_district, f)
                     for r in council.itertuples() for f, c in FIELDS.items() if getattr(r, c) == ""],
                    columns=["candidate_key", "candidate_name", "county_city", "electoral_district", "field"])
PID = {p["candidate_key"]: p["pid"] for p in people}
gaps.insert(0, "pid", gaps.candidate_key.map(PID))
assert gaps.pid.notna().all() and gaps.candidate_key.nunique() == 79 and len(gaps) == 208
print(gaps.field.value_counts().sort_index().to_dict())


{'G1': 69, 'G3': 69, 'G5': 70}


## 2. 中選會歷屆候選人檔比對

以姓名比對（原住民候選人另以漢名部分比對，因歷屆檔把族名接在姓名後），並把行政區代碼轉成名稱。是否為同一人在第 4 節逐人判斷。

In [3]:
geo = {}
for r in csv.reader((PROJECT_ROOT / INPUTS["cec_2022_geo_base"]).open(encoding="utf-8-sig")):
    geo[tuple(c.strip().lstrip("'") for c in r[:5])] = r[5]

def place(g):
    county = geo.get((g[0], g[1], "00", "000", "0000"), "")
    town = geo.get((g[0], g[1], "00", g[3], "0000"), "") if g[3] != "000" else ""
    vil = geo.get((g[0], g[1], "00", g[3], g[4]), "") if g[4] != "0000" else ""
    return county + town + vil

han = {p["pid"]: re.sub(r"[A-Za-z．.·‧\s]+", "", p["name"]) for p in people}
indigenous = {pid for pid, p in zip(han, people) if han[pid] != p["name"]}
hits, seen_paths = [], set()
for m in cec_manifest["members"]:
    if m["path"] in seen_paths:
        continue
    seen_paths.add(m["path"])
    ctx = m["archive_member"].split("/voteData/", 1)[-1].rsplit("/elcand.csv", 1)[0]
    with (PROJECT_ROOT / m["path"]).open(encoding="utf-8-sig") as fh:
        for line, row in enumerate(csv.reader(fh), 1):
            if len(row) <= 12:
                continue
            nm = row[6].strip()
            for pid, h in han.items():
                if nm == h or (pid in indigenous and nm.startswith(h)):
                    g = tuple(c.strip().lstrip("'") for c in row[:5])
                    hits.append(dict(pid=pid, cec_name=nm, election_context=ctx, geo_code="|".join(g), place=place(g),
                                     sex_code=row[8].strip().lstrip("'"), birth_raw=row[9].strip().lstrip("'"),
                                     education_cec=row[12].strip(), elected_marker=row[14].strip() if len(row) > 14 else "",
                                     source_file=m["path"], source_row=line))
scan = pd.DataFrame(hits)
print(scan.pid.nunique(), "people with name hits;", len(scan), "rows")


34 people with name hits; 82 rows


## 3. 判斷規則

沿用縣市首長補查的 G5 標準（明確敘述／中選會紀錄 → 0；自稱或報導「政治素人」「並無政治經歷」「首次參選」→ 1；擬參選人不算登記參選；中選會歷屆檔查無本人 → 1）。本輪另外：

1. **中選會紀錄的身分佐證**：同名紀錄在同一議員選區（或其內的里／區）、出生日與人物頁一致、或新聞／人物頁明載曾任該職，才視為本人；採用後可同時提供出生年（G1）、學歷級別（G3）與曾參選（G5=0）。另一縣市或年齡明顯不符者視為他人，對 G5 等同「查無本人」。同城市但無法確認者不採用，列入手動查核。
2. **報導年齡**：只有「今年XX歲」時，G1 記該年齡，`decision_basis=age_from_report`（與出生年算法最多差 1 歲）。
3. **G3 學歷**：明載畢業或學位者依級別；「肄業」「就讀」「攻讀」不算完成。候選人頁、維基學歷欄或政黨頁**只列校系、未寫畢業也未寫肄業**者，視為已完成，`decision_basis=listed_education_assumed_completed`（比照楊喬茵對洪方隆、張靜的決定；**待確認**）。
4. **查無學歷**：依楊喬茵「沒有任何學歷就歸最低」的決定，僅用於官方紀錄本身為「其他」或未填者（李文）；新聞查不到學歷的人仍留空。

In [4]:
def page_text(label):
    soup = BeautifulSoup((PROJECT_ROOT / CAPTURE[label]["file"]).read_bytes(), "html.parser")
    for t in soup(["script", "style", "noscript"]):
        t.decompose()
    return norm(soup.get_text(" "))
TEXT = {}
def has(label, phrase):
    if label not in TEXT:
        TEXT[label] = page_text(label)
    return norm(phrase) in TEXT[label]

# (pid, field, value, basis, [(label, phrase)], [CEC file:row], note)
D = []
def add(pid, field, value, basis, ev=(), cecs=(), note=""):
    D.append((pid, field, value, basis, list(ev), list(cecs), note))


In [5]:

def pt(i): return f"p{i:02d}_politicaltw"
def wk(i): return f"p{i:02d}_wiki"
def hit(pid, ctx_part, birth):
    h = scan[(scan.pid == pid) & scan.election_context.str.contains(ctx_part, regex=False) & scan.birth_raw.eq(birth)]
    assert len(h) >= 1, (pid, ctx_part, birth)
    return f"{h.iloc[-1].source_file}:{h.iloc[-1].source_row}"

# ---------- 臺北市 ----------
r = hit("p01", "2022", "0850716")
add("p01", "G1", "30", "cec_record_identity_confirmed", [(pt(1), "曾於2022年參選北投區裕民里里長")], [r], "中選會 2022 北投區裕民里紀錄出生 1996-07-16；political.tw 載明其 2022 參選裕民里長。")
add("p01", "G3", "大學以下", "source_rule", [(pt(1), "謝閔弘就讀國立臺灣師範大學東亞學系")], [r], "中選會 2022 學歷高中(職)；目前就讀大學不算完成。")
r = hit("p02", "2020總統立委", "0530922")
add("p02", "G5", "0", "cec_record_identity_confirmed", [(pt(2), "李敖女兒李文宣布參選士林北投議員")], [r], "中選會 2020 臺北市立委候選人李文（女，1964-09-22），與主表 G1=62（1964 年生）一致；political.tw 另載其 2022 宣布參選市長。")
add("p02", "G3", "大學以下", "user_rule_no_education_lowest", [], [r], "中選會 2020 學歷欄為「其他」，依楊喬茵「查無學歷歸最低」決定。")
add("p03", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "歷屆檔僅有屏東縣潮州鎮里長同名者（另一縣市）；political.tw 同名頁為新北民眾黨參選人，亦非本人。")
r = hit("p04", "2022", "0710116")
for f, v, n in [("G1", "44", "出生 1982-01-16"), ("G3", "大學", "2014／2018／2022 學歷皆為大學"), ("G5", "0", "2014、2018、2022 當選士林區後港里長")]:
    add("p04", f, v, "cec_record_identity_confirmed", [(pt(4), "中央社報導列紀建漢為臺北市士林區後港里里長"), ("ltn_441727", "後港里長候選人紀建漢")], [r], n)
add("p05", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "")
add("p06", "G1", "37", "source_rule", [(wk(6), "1989年4月10日")], [], "維基百科出生 1989-04-10。")
add("p07", "G1", "48", "source_rule", [(wk(7), "1978年10月10日"), ("ltn_5564523", "47歲劉致妤")], [], "維基百科出生 1978-10-10；2026-09 報導 47 歲（生日前）一致。")
add("p07", "G3", "大學以下", "source_rule", [(wk(7), "育達商職畢業"), ("ltn_5564523", "決定重返校園攻讀社工相關課程")], [], "最高已完成為商職；攻讀中不算完成。")
add("p07", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [("ltn_5564523", "決定從幕後助理走向第一線")], [], "2021 年曾參選新黨黨代表（黨內選舉，非公職）。")
r = hit("p08", "2022", "0670315")
for f, v, n in [("G1", "48", "出生 1978-03-15"), ("G3", "研究所", "學歷碩士"), ("G5", "0", "2014、2022 參選北投區文林里長")]:
    add("p08", f, v, "cec_record_same_district_name_match", [], [r], n + "；文林里位於本選區（士林北投），歷屆檔僅此一人同名。身分未經新聞獨立佐證，待確認。")
add("p09", "G1", "36", "source_rule", [(wk(9), "1990年8月19日")], [], "")
add("p09", "G3", "研究所", "source_rule", [(wk(9), "國立臺灣大學 政研所碩士")], [], "")
add("p09", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [("ltn_5559131", "五席裡面之中唯一的新人陳冠安")], [], "")
r = hit("p10", "2024", "0691017")
add("p10", "G3", "研究所", "cec_record_identity_confirmed", [(wk(10), "國立臺灣大學國家發展研究所")], [r], "中選會 2010–2024 學歷皆為碩士。")
add("p11", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "")
r = hit("p12", "2018", "0710524")
add("p12", "G5", "0", "cec_record_identity_confirmed", [], [r], "中選會 2018 臺北市第2選區議員候選人陳又新，出生 1982-05-24，與主表 G1=44 一致且同選區。")
r = hit("p13", "2024", "0620211")
for f, v, n in [("G1", "53", "出生 1973-02-11"), ("G3", "大學以下", "學歷專科"), ("G5", "0", "2008、2024 參選立委")]:
    add("p13", f, v, "cec_record_identity_confirmed", [(pt(13), "不要任何政黨在國會過半")], [r], n + "；2024 臺北市第4選區（內湖南港）立委候選人，與本選區相同；political.tw 載其 2024-01 立委選舉主張。")
add("p14", "G3", "研究所", "listed_education_assumed_completed", [("ltn_5384345", "鄧樂強是台大法律系畢業的空軍退役上校")], [], "台大法律系明載畢業；候選人頁另列「國立銘傳大學企業管理研究所」未寫畢業，依列名視為完成。")
add("p14", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "")
add("p15", "G1", "26", "age_from_report", [("ltn_5606134", "26歲新人郭凡")], [], "報導 26 歲（網友另稱 2000 年生）。")
add("p15", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [("ltn_5602783", "尤其郭凡是新人")], [], "")
add("p16", "G1", "33", "source_rule", [(wk(16), "1993年3月30日")], [], "")
r = hit("p16", "2024", "0820330")
add("p16", "G5", "0", "cec_record_identity_confirmed", [(wk(16), "1993年3月30日")], [r], "中選會 2024 高雄市立委候選人李明璇出生 1993-03-30，與維基百科完全一致。")
add("p17", "G3", "大學", "source_rule", [(wk(17), "畢業於 國立政治大學 新聞學系")], [], "")
add("p18", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "")
add("p19", "G1", "43", "source_rule", [(wk(19), "1983年11月15日")], [hit("p19", "2022", "0721115")], "")
add("p19", "G3", "研究所", "source_rule", [(wk(19), "政治大學 東亞研究所碩士")], [], "")
add("p19", "G5", "0", "cec_record_identity_confirmed", [(wk(19), "2022年曾代表時代力量參與臺北市第三選舉區議員選舉")], [hit("p19", "2022", "0721115")], "2018、2022 參選臺北市第3選區議員。")
add("p20", "G5", "1", "source_rule", [("ltn_5398484", "許原榮出來選舉，是大姑娘進洞房，是第一次")], [], "韓國瑜掃街時稱其第一次選舉。")
add("p21", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [("ltn_5461701", "朱政騏以新人之姿搶下提名席次")], [], "")
for p in ["p22", "p23", "p25"]:
    add(p, "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "")
r = hit("p24", "2022", "0680818")
for f, v, n in [("G1", "47", "出生 1979-08-18"), ("G3", "研究所", "學歷碩士"), ("G5", "0", "2022 當選大安區車層里長")]:
    add("p24", f, v, "cec_record_identity_confirmed", [(pt(24), "中央社報導列李易儒為臺北市大安區車層里里長")], [r], n + "。注意：自由時報另有同名的慈濟疫苗詐騙案通緝犯，與本候選人無關。")
r = hit("p26", "2018", "0490721")
for f, v, n in [("G1", "66", "出生 1960-07-21"), ("G3", "研究所", "學歷碩士"), ("G5", "0", "2018 參選臺北市第4選區議員")]:
    add("p26", f, v, "cec_record_identity_confirmed", [(pt(26), "自己多年來，深耕於中山大同")], [r], n + "；同選區（中山大同）。其他同名者在臺東、臺南，非本人。")
add("p27", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "")
r = hit("p28", "2020", "0730717")
add("p28", "G5", "0", "cec_record_identity_confirmed", [(wk(28), "1984年7月17日")], [r], "中選會 2020 新北市立委候選人張銘祐出生 1984-07-17，與維基百科一致。")
add("p28", "G3", "研究所", "listed_education_assumed_completed", [(wk(28), "台北海洋科技大學 海洋休閒觀光管理研究所")], [], "維基學歷欄只對北教大註明肄業，台北海洋科大研究所未註明肄業，視為完成。")
add("p29", "G1", "48", "age_from_report", [("ltn_1777175", "四十八歲在地傳產業者陳力新")], [], "")
add("p29", "G5", "1", "G5_reported_novice_rule_20261104", [("ltn_1777175", "以身作則鼓勵素人參政")], [], "以素人參政自居。")
add("p30", "G3", "大學", "listed_education_assumed_completed", [], [], "主表 education_raw「東海大學社會工作學系」，未註明畢業或肄業。")
add("p30", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "")
add("p31", "G5", "1", "source_rule", [("ltn_5555652", "這是她首度以候選人身分站到大眾面前")], [], "")
add("p32", "G1", "66", "source_rule", [(wk(32), "1960年4月")], [], "維基百科出生 1960 年 4 月。")
add("p32", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "")
add("p33", "G3", "大學", "listed_education_assumed_completed", [(wk(33), "長榮大學 大眾傳播學系")], [], "")
add("p33", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "")
for p in ["p34", "p35", "p37"]:
    add(p, "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "")
add("p36", "G3", "大學", "source_rule", [(wk(36), "並畢業於 輔仁大學 經濟學系")], [], "")
r = hit("p38", "2024", "0721223")
add("p38", "G1", "43", "source_rule", [(wk(38), "1983年12月23日")], [r], "")
add("p38", "G3", "研究所", "source_rule", [(wk(38), "淡江大學 中國大陸研究所法學碩士")], [], "")
add("p38", "G5", "0", "cec_record_identity_confirmed", [(wk(38), "2022年參選桃園市議員落選")], [r], "2022 桃園市議員、2024 桃園市立委候選人。注意：中選會已以設籍未滿 10 年不予登記 2026 參選。")
add("p39", "G1", "30", "source_rule", [(wk(39), "劉品妡 （1996年—）")], [], "維基百科 1996 年生。")
add("p39", "G3", "大學", "source_rule", [(wk(39), "劉品妡畢業於 東吳大學 政治學系")], [], "倫敦國王學院碩士為「攻讀」，未載完成。")
add("p41", "G3", "大學以下", "source_rule", [(wk(41), "服裝設計 學系（肄業）")], [], "大學肄業；最高已完成為高中。")
add("p41", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "")
r = hit("p42", "2022", "0720731")
for f, v, n in [("G1", "43", "出生 1983-07-31"), ("G3", "研究所", "學歷碩士"), ("G5", "0", "2022 以民眾黨參選新北市山地原住民議員")]:
    add("p42", f, v, "cec_record_identity_confirmed", [("newtalk_740515_lin_jia_hao", "民眾黨 新北市山地原住民市議員參選人 林家豪"), (pt(42), "有限責任新北市原住民電腦資訊勞動合作社理事主席")], [r], n + "。")
add("p43", "G1", "", "still_unknown", [], [], "中時搜尋結果摘要稱「李傅中武27歲女兒」，但未取得原文頁，未採用；已讀 ETtoday、自由時報徵召報導全文，均無年齡。")
for p in ["p43", "p44"]:
    add(p, "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "")
add("p40", "G5", "", "still_unknown", [], [], "中選會 2024 臺北市第8選區立委候選人張維學（1949 年生）與本選區重疊，但無法確認是否同一人。")
# ---------- 高雄市 ----------
add("p45", "G1", "63", "age_from_report", [("udn_9366552_zhong_bing_guang", "鍾炳光是警界出身，今年63歲")], [], "")
add("p45", "G3", "大學以下", "source_rule", [("udn_9366552_zhong_bing_guang", "警專第一期畢業")], [], "所載最高學歷為警專（專科）。")
add("p45", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [("udn_9366552_zhong_bing_guang", "自已已擔任公職41年")], [], "歷屆檔 2010 同名里長候選人 1952 年生，與今年 63 歲不符，非本人；公職為警察與區長（非民選）。")
add("p46", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [("ltn_5440841", "國民黨高雄市旗美地區最年輕市議員參選人李振豪")], [], "歷屆檔同名者在苗栗縣、1961 年生，非本人；新聞中竹聯幫同名者亦非本人。")
r = hit("p47", "2022", "0381102")
for f, v, n in [("G1", "77", "出生 1949-11-02"), ("G3", "大學以下", "學歷高中(職)"), ("G5", "0", "曾任茄萣鄉民代表，2018／2022 參選議員、2020 參選立委")]:
    add("p47", f, v, "cec_record_identity_confirmed", [("ltn_3045545", "李柏融（70歲）過去是高雄縣茄萣鄉鄉民代表")], [r], n + "；同選區（茄萣等）。")
add("p48", "G3", "研究所", "source_rule", [("ltn_5334889", "琳潔是英國倫敦大學公共行政碩士畢業")], [], "")
add("p48", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "")
add("p49", "G3", "研究所", "source_rule", [("ltn_5518335", "擁有高雄科技大學水產科技產業博士學位")], [], "")
add("p49", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "")
add("p50", "G1", "35", "source_rule", [("ltn_5518335", "蔡秉璁80年次")], [], "民國 80 年 = 1991。")
add("p50", "G3", "研究所", "source_rule", [("ltn_5518335", "再取得清華大學社會所碩士學位")], [], "")
r = hit("p51", "2020", "0730104")
add("p51", "G1", "42", "source_rule", [(wk(51), "1984年1月4日")], [r], "")
add("p51", "G3", "研究所", "source_rule", [("ltn_5472000", "義守大學管理碩士")], [], "")
add("p51", "G5", "0", "cec_record_identity_confirmed", [(wk(51), "2016年、2020年，兩度代表中國國民黨參選")], [r], "")
add("p52", "G1", "83", "age_from_report", [("ltn_5563312", "83歲阿嬤蔣黃女今天登記參選")], [], "")
add("p52", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "")
add("p53", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "歷屆檔同名者在新竹縣、彰化縣，非本人。")
add("p54", "G5", "1", "source_rule", [("ltn_5594781", "直言自己是第一次投入選舉的新人")], [], "")
add("p55", "G3", "研究所", "listed_education_assumed_completed", [], [], "主表 education_raw 列「大同大學工業設計研究所」，未註明畢業或肄業。")
add("p55", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "")
add("p56", "G3", "大學", "listed_education_assumed_completed", [], [], "主表 education_raw 列「中興大學」，未註明畢業或肄業。")
add("p56", "G5", "0", "source_rule", [("ltn_5466087", "時隔20餘年再次投入選戰")], [], "本人稱再次投入選戰，即曾參選；歷屆檔未見其名（可能為較早或補選）。")
r = hit("p57", "2018", "0480128")
for f, v, n in [("G1", "67", "出生 1959-01-28"), ("G3", "研究所", "學歷碩士"), ("G5", "0", "曾任 4 屆議員")]:
    add("p57", f, v, "cec_record_identity_confirmed", [("ltn_5236417", "曾任4屆議員的黃淑美")], [r], n + "；同選區（三民）。")
add("p58", "G5", "", "still_unknown", [], [], "中選會 2018 高雄市第11選區議員候選人余聖明（1972 年生）與本人選區不同，無法確認是否同一人。")
for p in ["p59", "p60", "p61", "p64", "p67", "p73", "p76"]:
    add(p, "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "")
r = hit("p62", "5屆立委", "047")
add("p62", "G1", "68", "cec_record_identity_confirmed", [("ltn_5562721", "今年68歲的鄒約翰是退役軍官，曾參與第5屆高雄區域立委選舉")], [r], "中選會出生民國 47 年 = 1958，與報導 68 歲一致。")
add("p62", "G5", "0", "cec_record_identity_confirmed", [("ltn_5562721", "曾參與第5屆高雄區域立委選舉")], [r], "")
add("p63", "G1", "55", "age_from_report", [("ltn_5563213", "黃鈺婷今年55歲")], [], "維基百科同名頁為啦啦隊員，非本人。")
add("p63", "G3", "研究所", "source_rule", [("ltn_5563213", "日本東京農業大學博士")], [], "")
add("p63", "G5", "1", "G5_no_prior_cec_record_rule_20261104", [], [], "")
add("p65", "G5", "", "still_unknown", [], [], "中選會 2014 高雄市議員、2024 高雄市立委有同名者（1956 年生），報導未提其參選史，無法確認。")
add("p66", "G5", "", "still_unknown", [], [], "中選會 2020 高雄市第6選區立委有同名女性（1974 年生），無法確認是否同一人。")
add("p68", "G5", "0", "source_rule", [("ltn_5435219", "李紘毅是現任高雄市小港區港明里的里長")], [], "現任民選里長。歷屆檔港明里長名為李文正（1956 年生），可能為改名，未確認，故不採其出生年與學歷。")
r = hit("p69", "2022", "0741118")
add("p69", "G1", "41", "cec_record_identity_confirmed", [("ltn_5561741", "這是她繼2018、2022之後，第三度登記參選前鎮小港市議員")], [r], "中選會 2018／2022 同選區紀錄出生 1985-11-18。")
add("p69", "G3", "大學", "source_rule", [("ltn_5387236", "她介紹自己畢業於成大護理系")], [], "")
add("p70", "G1", "38", "age_from_report", [("ltn_5562721", "張藝璉今年38歲")], [], "")
add("p70", "G3", "研究所", "source_rule", [("ltn_5562721", "中央大學法政所碩士")], [], "")
add("p70", "G5", "1", "G5_reported_novice_rule_20261104", [("ltn_5562721", "第五屆高市議員選舉有多位「素人」登記")], [], "報導以素人介紹，本人稱「至少要嘗試一次」。")
add("p71", "G5", "1", "source_rule", [("ltn_5610319", "這是他第一次參選")], [], "維基百科／political.tw 同名者為前臺中市議員，非本人。")
for pid, b, g1, g3, ctx in [("p72", "0600213", "55", "大學", "2022"), ("p74", "0560608", "59", "大學", "2022"), ("p75", "0530525", "62", "研究所", "2022")]:
    r = hit(pid, ctx, b)
    lab = pt(int(pid[1:]))
    add(pid, "G1", g1, "cec_record_identity_confirmed", [(lab, "2022年高雄市第1")], [r], "political.tw 載其為 2022 同選區議員候選人。")
    add(pid, "G3", g3, "cec_record_identity_confirmed", [(lab, "2022年高雄市第1")], [r], "")
    add(pid, "G5", "0", "cec_record_identity_confirmed", [(lab, "2022年高雄市第1")], [r], "")
add("p73", "G3", "大學", "listed_education_assumed_completed", [], [], "主表 education_raw 列「正修科技大學-休閒運動與管理學系」，未註明畢業或肄業。")
r = hit("p77", "2022", "0610218")
for f, v, n in [("G1", "54", "出生 1972-02-18"), ("G3", "大學以下", "學歷高中(職)以下"), ("G5", "0", "2022 當選高雄市第14選區議員")]:
    add("p77", f, v, "cec_record_identity_confirmed", [("ltn_5130274", "高雄市議員高忠德")], [r], n + "（歷屆檔姓名含族名 Takiludun．Anu）。")
r = hit("p79", "2022", "0600513")
for f, v, n in [("G1", "55", "出生 1971-05-13"), ("G3", "研究所", "2022 學歷碩士"), ("G5", "0", "2014–2022 當選茂林區民代表")]:
    add("p79", f, v, "cec_record_same_district_name_match", [(pt(79), "居住茂林區")], [r], n + "；茂林區屬本選區（山地原住民南區），歷屆檔僅此一人同名。")

for pid, field, value, basis, ev, cecs, note in D:
    for label, phrase in ev:
        assert label in CAPTURE, (pid, label)
        assert has(label, phrase), (pid, field, label, phrase)
print(len(D), "decisions;", sum(len(d[4]) for d in D), "source phrases and", sum(len(d[5]) for d in D), "CEC rows verified")


142 decisions; 105 source phrases and 56 CEC rows verified


## 5. 組成補查紀錄與手動查核清單

In [6]:
dec = {}
for pid, field, value, basis, ev, cecs, note in D:
    assert (pid, field) not in dec, (pid, field)
    dec[(pid, field)] = (value, basis, ev, cecs, note)
gap_keys = set(zip(gaps.pid, gaps.field))
extra = set(dec) - gap_keys
assert not extra, f"decisions for non-gaps: {sorted(extra)}"
URL = {l: c["url"] for l, c in CAPTURE.items()}
FILE = {l: c["file"] for l, c in CAPTURE.items()}
rows = []
for g in gaps.itertuples():
    value, basis, ev, cecs, note = dec.get((g.pid, g.field), ("", "still_unknown", [], [], ""))
    if basis == "still_unknown" and not note:
        note = "全文閱讀後仍未找到可用資料。"
    rows.append(dict(candidate_key=g.candidate_key, pid=g.pid, candidate_name=g.candidate_name, county_city=g.county_city,
                     electoral_district=g.electoral_district, field=g.field, current_value="", proposed_value=value,
                     decision="accepted" if value else "still_unknown", decision_basis=basis if value else "",
                     check_method="手動查核" if value else "",
                     evidence_excerpt=" ｜ ".join(p for _, p in ev), source_url=" ｜ ".join(dict.fromkeys(URL[l] for l, _ in ev)),
                     source_file=" ｜ ".join(dict.fromkeys(FILE[l] for l, _ in ev)), cec_record=" ｜ ".join(cecs),
                     note=note, checked_date="2026-11-04"))
review = pd.DataFrame(rows)
assert len(review) == 208 and not review.duplicated(["candidate_key", "field"]).any()
print(pd.crosstab(review.field, review.decision, margins=True))
print(review[review.decision.eq("accepted")].decision_basis.value_counts().to_dict())

scan["candidate_name"] = scan.pid.map({p["pid"]: p["name"] for p in people})
used = {c.split(":")[0] + ":" + c.split(":")[1] for cs in review.cec_record for c in cs.split(" ｜ ") if c}
scan["used_as_identity_evidence"] = [f"{f}:{r}" in used for f, r in zip(scan.source_file, scan.source_row)]

open_rows = review[review.decision.eq("still_unknown")]
search_url = lambda n: "https://search.ltn.com.tw/list?keyword=" + n
links = []
for r in open_rows.itertuples():
    name = re.sub(r"[A-Za-z．.·‧\s]+", "", r.candidate_name)
    links.append(dict(candidate_key=r.candidate_key, candidate_name=r.candidate_name, county_city=r.county_city,
                      electoral_district=r.electoral_district, field=r.field, concern=r.note,
                      political_tw="https://political.tw/people/" + name + "/", ltn_search=search_url(name),
                      cec_hits="；".join(f"{h.election_context} {h.place} 出生{h.birth_raw}" for h in scan[scan.candidate_name.eq(r.candidate_name)].itertuples())))
links = pd.DataFrame(links)
print(len(links), "open gaps listed for manual check")


decision  accepted  still_unknown  All
field                                 
G1              32             37   69
G3              39             30   69
G5              66              4   70
All            137             71  208
{'cec_record_identity_confirmed': 45, 'G5_no_prior_cec_record_rule_20261104': 37, 'source_rule': 33, 'listed_education_assumed_completed': 7, 'age_from_report': 6, 'cec_record_same_district_name_match': 6, 'G5_reported_novice_rule_20261104': 2, 'user_rule_no_education_lowest': 1}


71 open gaps listed for manual check


In [7]:
outputs = {"gap_review": review, "cec_name_scan": scan, "manual_check_links": links}
verification = {}
for key, frame in outputs.items():
    path = PROJECT_ROOT / OUTPUTS[key]
    with path.open("x", encoding="utf-8-sig", newline="") as stream:
        frame.to_csv(stream, index=False, lineterminator="\n")
    reread = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    assert list(reread.columns) == list(frame.columns) and len(reread) == len(frame)
    pd.testing.assert_frame_equal(frame.astype(str).reset_index(drop=True), reread, check_dtype=False)
    verification[key] = dict(path=OUTPUTS[key], rows=len(reread), columns=len(reread.columns), sha256=sha(OUTPUTS[key]))
print(json.dumps(verification, ensure_ascii=False, indent=1))


{
 "gap_review": {
  "path": "data/processed/council_tk_g1_g5_gap_review_20261104.csv",
  "rows": 208,
  "columns": 17,
  "sha256": "ff0233b49349a37616895981fb797f929ed8512c2afdab874b041237aa39a82a"
 },
 "cec_name_scan": {
  "path": "data/processed/council_tk_cec_name_scan_20261104.csv",
  "rows": 82,
  "columns": 13,
  "sha256": "42192cdcf18b14790d63a005b7e07fbcf7e00619e86c874d954b917de7b0c888"
 },
 "manual_check_links": {
  "path": "data/processed/council_tk_manual_check_links_20261104.csv",
  "rows": 71,
  "columns": 9,
  "sha256": "ea715bf02cc7be0089a12c602634817bfb0d08106de7aa1b652575b00478c0a7"
 }
}


## 6. 若併入補上值，北高議員缺值變化（僅在記憶體試算，未寫回主表）

In [8]:
summary = pd.DataFrame({f: {"缺值（目前）": (council[c] == "").sum(),
                            "補上": ((review.field == f) & review.decision.eq("accepted")).sum(),
                            "其中：待確認的推定規則": ((review.field == f) & review.decision_basis.isin(["listed_education_assumed_completed", "cec_record_same_district_name_match"])).sum(),
                            "仍缺": ((review.field == f) & review.decision.eq("still_unknown")).sum()}
                        for f, c in FIELDS.items()}).T
summary


,缺值（目前）,補上,其中：待確認的推定規則,仍缺
G1,69,32,2,37
G3,69,39,9,30
G5,70,66,2,4
